# CellChat prep — label CD4 T cells by TLS radial category

Prepare the AnnData object for CellChat ligand–receptor analysis across TLS sub-regions. TLS-associated
cells (those with a `tls_radial_category` from the radial-distance analysis) are kept, and the activated
CD4 T-cell subsets are collapsed into a single label per region — `CD4 act (tls core | tls outer | tls surround)` —
so CellChat can compare signaling involving activated CD4 cells at different radial positions within the TLS.
The prepped object is written to disk for the R CellChat notebook.

**Pinned Environment:** [`conda_envs/space2_20250604.yml`](../../conda_envs/space2_20250604.yml)

In [1]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
import scanpy as sc

warnings.filterwarnings("ignore")

## Local file info

In [2]:
from pathlib import Path
sys.path.append(str(Path.cwd().resolve().parents[1]))
from config.paths import BASE_OUTDIR

out_dir = os.path.join(BASE_OUTDIR, "downstream_analysis")
out_dir_cellchat = os.path.join(out_dir, "cellchat_tls-category")

if not os.path.exists(out_dir_cellchat):
    os.makedirs(out_dir_cellchat)

plot_out_dir = os.path.join(out_dir_cellchat, 'plots')
if not os.path.exists(plot_out_dir):
    os.makedirs(plot_out_dir)

## Load adata

Load the distance/structure-labeled object from the radial-distance analysis
(`adata_distance_zones_structure_tls_dist_categories.h5ad`), which already carries a
`tls_radial_category` (tls core / outer / surround) per TLS-associated cell.

In [3]:
adata = sc.read_h5ad(os.path.join(out_dir,'adata_distance.h5ad'))

## Relabel activated CD4 T cells by TLS radial category

Subset to TLS-associated cells (those with a non-null `tls_radial_category`), then collapse the
activated CD4 subsets (`Th0`, `Th1`, `Th17`, `Th2`, `Treg`, `CD4 trans`) into a single
`CD4 act (<radial category>)` label. This lets CellChat treat activated CD4 cells in the TLS
**core**, **outer**, and **surround** as distinct cell groups when scoring ligand–receptor
interactions.

In [4]:
# Categorize by tls-category 
adata = adata[~adata.obs['tls_radial_category'].isna(), :]
# set labels
adata.obs['tls_radial_category'] = adata.obs['tls_radial_category'].values.tolist()
adata.obs['label_fine'] = adata.obs['label_fine'].values.tolist()

In [5]:
#wherever cell_type_1 is 'T Cells', add the classification to the cell_type_1 column
# Create a mask for CD4 act
cd4_mask = adata.obs['label_fine'].isin(['Th0', 'Th1', 'Th17', 'Th2', 'Treg', 'CD4 trans'])

# make single activated T cell label per region 
adata.obs.loc[cd4_mask, 'label_fine'] = (
    'CD4 act (' + adata.obs.loc[cd4_mask, 'tls_radial_category'] + ')'
)

# Check the updated cell types
print("Updated T cell categories:")
print(adata.obs.loc[cd4_mask, 'label_fine'].value_counts())

# adata.obs['label_medium']


Updated T cell categories:
label_fine
CD4 act (tls core)        693
CD4 act (tls outer)       636
CD4 act (tls surround)    565
Name: count, dtype: int64


In [9]:
# #wherever cell_type_1 is 'T Cells', add the classification to the cell_type_1 column
# # Create a mask for CD4 act
# cd4_mask = adata.obs['label_fine'].isin(['Th0', 'Th1', 'Th17', 'Th2', 'Treg', 'CD4 trans'])

# # make single activated T cell label per region 
# adata.obs.loc[cd4_mask, 'label_fine'] = (
#     'CD4 act (' + adata.obs.loc[cd4_mask, 'tls_radial_category'] + ')'
# )

# # Check the updated cell types
# print("Updated T cell categories:")
# print(adata.obs.loc[cd4_mask, 'label_fine'].value_counts())

# # adata.obs['label_medium']


Updated T cell categories:
label_fine
CD4 act (tls core)        725
CD4 act (tls outer)       604
CD4 act (tls surround)    565
Name: count, dtype: int64


In [6]:
# Check the updated cell types
adata_d3 = adata[adata.obs['sample_label']=='HDM_day3', :]
adata_d30 = adata[adata.obs['sample_label']=='HDM_day30', :]
print("Updated T cell categories, day 3:")
# cd4_mask = adata_d3.obs['label_fine'].isin(['CD4 act (tls core)', 'CD4 act (tls outer)', 'CD4 act (tls surround)'])
# print(adata_d3.obs.loc[cd4_mask, 'label_fine'].value_counts())

print("Updated T cell categories, day 30:")
# cd4_mask = adata_d30.obs['label_fine'].isin(['CD4 act (tls core)', 'CD4 act (tls outer)', 'CD4 act (tls surround)'])
# print(adata_d30.obs.loc[cd4_mask, 'label_fine'].value_counts())

Updated T cell categories, day 3:
Updated T cell categories, day 30:


## Save prepped adata

Write the relabeled object for the R CellChat notebook
([`01_cellchat_R_tls-category.ipynb`](01_cellchat_R_tls-category.ipynb)).

In [7]:
adata.write_h5ad(os.path.join(out_dir_cellchat,'adata_cellchat_prepped.h5ad'))